# Text Classification Evaluation Notebook

This notebook evaluates the accuracy of text descriptions generated by Gemini models and analyzes the correlation between accuracy and preserved semantic distances across categories.

## Workflow:
1. Load data from JSON (categories and texts).
2. Split data 80% training / 20% testing.
3. Generate descriptions using Gemini 3.1 Flash lite.
4. Classify training data in randomized batches of 20.
5. Evaluate performance with charts and tables.
6. Measure semantic distance preservation.
7. Analyze correlation between accuracy and semantic distance.
8. Ablation study with Gemini 3.5 Flash.

## 1. Setup and Configuration
Install necessary libraries and configure the Gemini API.

In [ ]:
!pip install -q google-genai pandas numpy matplotlib seaborn scikit-learn scipy

import os
import json
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google import genai
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr

# Configuration
CONFIG = {
    'MODEL_31': 'gemini-3.1-flash-lite-latest',
    'MODEL_35': 'gemini-3.5-flash-latest',
    'OUTPUT_DIR': 'outputs',
    'REUSE_CACHE': True,
    'API_KEY': os.getenv('GOOGLE_API_KEY')
}

if not os.path.exists(CONFIG['OUTPUT_DIR']):
    os.makedirs(CONFIG['OUTPUT_DIR'])

client = genai.Client(api_key=CONFIG['API_KEY'])

## 2. Data Loading
Mount Google Drive and load the dataset from the specified path.

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab or drive mount failed.')

def load_data(file_path):
    if not os.path.exists(file_path):
        print(f'Warning: File not found at {file_path}. Using dummy data.')
        return {
            'technology': ['Computers are fast', 'Software is evolving', 'AI is the future', 'Hardware improves over time'],
            'sports': ['Football is popular', 'Basketball requires skill', 'Tennis is played on grass or clay', 'Running is good exercise'],
            'cooking': ['Baking requires precision', 'Spices add flavor', 'Italian cuisine is famous', 'Vegetables are healthy']
        }
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

DATA_PATH = '/content/drive/MyDrive/Graphiko/exports/base_data/latest/channel_titles.json'
data = load_data(DATA_PATH)

## 3. Data Splitting
Split data 80% training and 20% testing per category.

In [ ]:
train_data = {}
test_data = {}

for category, texts in data.items():
    train, test = train_test_split(texts, test_size=0.2, random_state=42)
    train_data[category] = train
    test_data[category] = test

print(f'Categories: {list(data.keys())}')

## 4. LLM Caching Utility
Save and load LLM responses to avoid hitting quotas.

In [ ]:
def get_cache_path(name):
    return os.path.join(CONFIG['OUTPUT_DIR'], f'{name}.json')

def save_cache(name, data):
    with open(get_cache_path(name), 'w') as f:
        json.dump(data, f)

def load_cache(name):
    path = get_cache_path(name)
    if CONFIG['REUSE_CACHE'] and os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return None

## 5. Classification Helper Function
A helper function to classify texts in batches using category descriptions.

In [ ]:
def classify_texts(texts_with_labels, descriptions, model_name, cache_name):
    results = load_cache(cache_name) or []
    
    # Identify which texts still need classification
    processed_texts = {r['text'] for r in results}
    to_process = [t for t in texts_with_labels if t['text'] not in processed_texts]
    
    if not to_process:
        print(f'All texts already classified for {cache_name}.')
        return pd.DataFrame(results)
    
    batch_size = 20
    for i in range(0, len(to_process), batch_size):
        batch = to_process[i:i+batch_size]
        
        desc_str = '\n'.join([f'{cat}: {desc}' for cat, desc in descriptions.items()])
        texts_to_classify = '\n'.join([f"[{j}] {item['text']}" for j, item in enumerate(batch)])
        
        prompt = f'Classify the following texts into one of these categories:\n{desc_str}\n\nTexts:\n{texts_to_classify}\n\nReturn only a JSON list of category names in order.'
        
        try:
            response = client.models.generate_content(model=model_name, contents=prompt)
            res_text = response.text.strip()
            if '```json' in res_text: res_text = res_text.split('```json')[1].split('```')[0]
            predictions = json.loads(res_text)
            
            for j, pred in enumerate(predictions):
                if j < len(batch):
                    batch[j]['prediction'] = pred
                    results.append(batch[j])
            
            # Incremental save
            save_cache(cache_name, results)
            print(f'Processed batch {i//batch_size + 1}/{(len(to_process)-1)//batch_size + 1}')
        except Exception as e:
            print(f'Error processing batch starting at {i}: {e}')
            
    return pd.DataFrame(results)

## 6. Iterative Description Generation
Generate descriptions for each category using Gemini 3.1 Flash lite.

In [ ]:
descriptions_31 = load_cache('descriptions_31') or {}

for category, texts in train_data.items():
    if category in descriptions_31:
        continue
    
    prompt = f'Provide a concise description of the following category based on these examples: {category}\nExamples:\n' + '\n'.join(texts)
    response = client.models.generate_content(model=CONFIG['MODEL_31'], contents=prompt)
    descriptions_31[category] = response.text
    save_cache('descriptions_31', descriptions_31)
    
print('Descriptions generated for all categories.')

## 7. Batch Classification
Classify training data in randomized batches of 20 using the generated descriptions.

In [ ]:
all_train_texts = []
for category, texts in train_data.items():
    for t in texts:
        all_train_texts.append({'text': t, 'label': category})
random.seed(42)
random.shuffle(all_train_texts)

df_results = classify_texts(all_train_texts, descriptions_31, CONFIG['MODEL_31'], 'classification_results_31')
print('Classification complete.')

## 8. Performance Evaluation Helper
Helper to print metrics and show charts.

In [ ]:
def evaluate_performance(df, title_suffix=''):
    accuracy = accuracy_score(df['label'], df['prediction'])
    print(f'Global Accuracy {title_suffix}: {accuracy:.4f}')

    report = classification_report(df['label'], df['prediction'], output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    display(df_report)

    # Confusion Matrix
    plt.figure(figsize=(10, 8))
    categories = sorted(df['label'].unique())
    cm = confusion_matrix(df['label'], df['prediction'], labels=categories)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=categories, yticklabels=categories)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix {title_suffix}')
    plt.show()
    return accuracy

evaluate_performance(df_results, '(Gemini 3.1 Flash lite)')

## 9. Semantic Distance Preservation Metric
Evaluate how distances across categories were preserved compared to distribution of the texts.

In [ ]:
def get_embedding(text):
    response = client.models.embed_content(model='text-embedding-004', contents=text)
    return response.embeddings[0].values

def calculate_preservation(train_data, descriptions):
    # 1. Calculate centroid embeddings for each category from raw texts
    category_centroids = {}
    for category, texts in train_data.items():
        embeddings = [get_embedding(t) for t in texts]
        category_centroids[category] = np.mean(embeddings, axis=0)

    # 2. Calculate distance matrix for categories (raw texts)
    categories = sorted(list(train_data.keys()))
    dist_raw = np.zeros((len(categories), len(categories)))
    for i, cat1 in enumerate(categories):
        for j, cat2 in enumerate(categories):
            dist_raw[i, j] = cosine(category_centroids[cat1], category_centroids[cat2])

    # 3. Calculate distance matrix for category descriptions
    desc_embeddings = {cat: get_embedding(desc) for cat, desc in descriptions.items()}
    dist_desc = np.zeros((len(categories), len(categories)))
    for i, cat1 in enumerate(categories):
        for j, cat2 in enumerate(categories):
            dist_desc[i, j] = cosine(desc_embeddings[cat1], desc_embeddings[cat2])

    # Metric: Correlation between dist_raw and dist_desc
    raw_flat = dist_raw[np.triu_indices(len(categories), k=1)]
    desc_flat = dist_desc[np.triu_indices(len(categories), k=1)]
    preservation_score, _ = pearsonr(raw_flat, desc_flat)
    return preservation_score, dist_raw, dist_desc, categories

pres_score, dist_raw, dist_desc, categories = calculate_preservation(train_data, descriptions_31)
print(f'Semantic Distance Preservation Score: {pres_score:.4f}')

## 10. Correlation Analysis
Determine if there is a correlation between text description accuracy and preserved semantic distance.

In [ ]:
def analyze_correlation(df_results, dist_raw, dist_desc, categories):
    # Accuracy per category
    cat_accuracy = df_results.groupby('label').apply(lambda x: accuracy_score(x['label'], x['prediction']))

    # Distance preservation per category (mean correlation of its distances to others)
    cat_preservation = {}
    for i, cat in enumerate(categories):
        cat_preservation[cat] = pearsonr(dist_raw[i], dist_desc[i])[0]

    analysis_df = pd.DataFrame({
        'Accuracy': cat_accuracy,
        'Preservation': pd.Series(cat_preservation)
    })

    plt.figure(figsize=(8, 6))
    sns.regplot(data=analysis_df, x='Preservation', y='Accuracy')
    plt.title('Correlation: Description Accuracy vs Semantic Preservation')
    plt.show()

    corr_val, p_val = pearsonr(analysis_df['Preservation'], analysis_df['Accuracy'])
    print(f'Pearson Correlation: {corr_val:.4f} (p={p_val:.4f})')

    print('\nFinal Conclusion:')
    if corr_val > 0.5 and p_val < 0.05:
        print('There IS a significant positive correlation. Higher preservation of semantic distances leads to better classification accuracy.')
    else:
        print('There is NO strong evidence of correlation in this dataset. Accuracy may be driven by other factors like description clarity or category distinctness.')
    return corr_val

analyze_correlation(df_results, dist_raw, dist_desc, categories)

## 11. Ablation Study: Gemini 3.5 Flash (Single Request)
Compare results with descriptions generated by Gemini 3.5 Flash in a single request.

In [ ]:
ablation_descriptions = load_cache('ablation_descriptions')

if not ablation_descriptions:
    # Generate ALL descriptions in a single request
    all_train_formatted = json.dumps(train_data)
    prompt = f'Generate concise descriptions for ALL the following categories based on their examples:\n{all_train_formatted}\nReturn a JSON object where keys are category names and values are descriptions.'
    
    response = client.models.generate_content(model=CONFIG['MODEL_35'], contents=prompt)
    res_text = response.text.strip()
    if '```json' in res_text: res_text = res_text.split('```json')[1].split('```')[0]
    ablation_descriptions = json.loads(res_text)
    save_cache('ablation_descriptions', ablation_descriptions)

print('Ablation descriptions generated.')

# Run Classification for Ablation
df_results_ablation = classify_texts(all_train_texts, ablation_descriptions, CONFIG['MODEL_35'], 'classification_results_35')

# Evaluate Ablation
evaluate_performance(df_results_ablation, '(Gemini 3.5 Flash - Ablation)')

# Preservation Score for Ablation
pres_score_ab, dist_raw_ab, dist_desc_ab, cats_ab = calculate_preservation(train_data, ablation_descriptions)
print(f'Ablation Semantic Distance Preservation Score: {pres_score_ab:.4f}')

# Correlation for Ablation
analyze_correlation(df_results_ablation, dist_raw_ab, dist_desc_ab, cats_ab)